# 09. ALD 공정 데이터 EDA

더미 데이터로 파이프라인 검증. 실제 문헌 데이터 수집 후 `data/ald_process_data.csv` 교체 시 재실행.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

def gpc_model(material, temp, pulse_time):
    """ALD 물리를 반영한 더미 GPC 생성."""
    params = {
        'HfO2':  {'window': (150, 300), 'base_gpc': 1.1},
        'Al2O3': {'window': (100, 280), 'base_gpc': 1.05},
        'ZrO2':  {'window': (150, 280), 'base_gpc': 1.0},
    }
    p = params[material]
    lo, hi = p['window']
    gpc = p['base_gpc']
    if temp < lo:
        gpc *= (0.5 + 0.5 * (temp / lo))
    elif temp > hi:
        gpc *= (1 + 0.01 * (temp - hi))
    if pulse_time < 0.1:
        gpc *= (pulse_time / 0.1)
    return round(gpc + np.random.normal(0, 0.05), 3)

records = []
configs = {
    'HfO2':  {'precursors': ['TDMAH', 'HfCl4', 'TEMAHf'], 'oxidants': ['H2O', 'O3'],  'temps': (100, 380)},
    'Al2O3': {'precursors': ['TMA'],                        'oxidants': ['H2O', 'O3'],  'temps': (80,  320)},
    'ZrO2':  {'precursors': ['ZrCl4', 'TDMAZ'],            'oxidants': ['H2O', 'O3'],  'temps': (120, 330)},
}

for material, cfg in configs.items():
    n = 70 if material == 'HfO2' else 60 if material == 'Al2O3' else 40
    for _ in range(n):
        temp      = np.random.randint(*cfg['temps'])
        pulse     = round(np.random.choice([0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0]), 2)
        purge     = round(float(np.random.choice([5, 10, 15, 20, 30])), 1)
        precursor = np.random.choice(cfg['precursors'])
        oxidant   = np.random.choice(cfg['oxidants'])
        gpc       = gpc_model(material, temp, pulse)
        records.append({
            'material': material, 'precursor': precursor, 'oxidant': oxidant,
            'temperature': temp, 'pulse_time': pulse, 'purge_time': purge,
            'gpc': gpc, 'doi': 'DUMMY_DATA'
        })

df = pd.DataFrame(records)
print(f"더미 데이터: {len(df)}행")
print(df['material'].value_counts())
df.to_csv('../data/ald_process_data.csv', index=False)
print("저장 완료: ../data/ald_process_data.csv")
print("\n[주의] 이 데이터는 더미입니다. doi='DUMMY_DATA' 행을 실제 문헌 데이터로 교체하세요.")

더미 데이터: 170행
material
HfO2     70
Al2O3    60
ZrO2     40
Name: count, dtype: int64
저장 완료: ../data/ald_process_data.csv

[주의] 이 데이터는 더미입니다. doi='DUMMY_DATA' 행을 실제 문헌 데이터로 교체하세요.


In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/ald_process_data.csv')
print(f"데이터: {len(df)}행")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. GPC 분포 (소재별)
for mat, grp in df.groupby('material'):
    axes[0,0].hist(grp['gpc'], bins=15, alpha=0.6, label=mat)
axes[0,0].set_xlabel('GPC (Å/cycle)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('GPC Distribution by Material')
axes[0,0].legend()

# 2. Temperature vs GPC
colors = {'HfO2': 'steelblue', 'Al2O3': 'darkorange', 'ZrO2': 'seagreen'}
for mat, grp in df.groupby('material'):
    axes[0,1].scatter(grp['temperature'], grp['gpc'],
                      alpha=0.6, s=20, color=colors[mat], label=mat)
axes[0,1].set_xlabel('Temperature (°C)')
axes[0,1].set_ylabel('GPC (Å/cycle)')
axes[0,1].set_title('Temperature vs GPC — ALD Window 확인')
axes[0,1].legend()

# 3. Pulse time vs GPC
axes[1,0].scatter(df['pulse_time'], df['gpc'], alpha=0.5, s=20, color='purple')
axes[1,0].set_xlabel('Pulse Time (s)')
axes[1,0].set_ylabel('GPC (Å/cycle)')
axes[1,0].set_title('Pulse Time vs GPC — Saturation 확인')

# 4. Precursor 분포
df['precursor'].value_counts().plot(kind='bar', ax=axes[1,1], color='steelblue')
axes[1,1].set_xlabel('Precursor')
axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Precursor Distribution')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../data/ald_eda.png', dpi=150)
plt.close()
print("저장 완료: ../data/ald_eda.png")

print(f"\n=== GPC 통계 ===")
print(df.groupby('material')['gpc'].describe().round(3))
suspicious = df[df['gpc'] > 4.0]
print(f"\nGPC > 4.0 (CVD 모드 의심): {len(suspicious)}개")

데이터: 170행
저장 완료: ../data/ald_eda.png

=== GPC 통계 ===
          count   mean    std    min    25%    50%    75%    max
material                                                        
Al2O3      60.0  0.995  0.261  0.430  0.992  1.054  1.116  1.424
HfO2       70.0  1.159  0.298  0.481  1.056  1.110  1.211  2.030
ZrO2       40.0  0.940  0.259  0.434  0.910  0.980  1.030  1.549

GPC > 4.0 (CVD 모드 의심): 0개
